# 2. Baseline modeling — 3D U-Net + transformer

Thin driver around the vendored baseline in `scripts/` (see
`docs/0_coding_standards.md` for why that logic lives in `scripts/` rather
than `src/` for now). Two independent things this notebook can do,
controlled by `RUN_MODE`:

- **`"submission"`**: predict on the real competition `test/` set and write
  `submission.csv`, for upload. Defaults to the baseline author's public
  pretrained checkpoint (`thibautgoldsborough/cellmot-baseline-artifacts`)
  so a first submission doesn't require training anything ourselves —
  see `docs/1_instructions.md`.
- **`"train"`**: train our own checkpoint from scratch (a documented next
  experiment, not required for a first submission).

Not yet run: needs the competition data, which isn't downloaded locally —
run on Kaggle via `scripts/push_kaggle_kernel.sh baseline` (competition
mount + both Kaggle Dataset sources auto-detect, see the Setup cell) or
point `$CELLMOT_DATA_DIR` at a local copy.

## 1. Setup & Config

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()


def _find_mount(candidates: list[Path], marker: str) -> Path | None:
    """Return the first candidate containing ``marker``, else scan /kaggle/input."""
    for c in candidates:
        if (c / marker).exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob(f"**/{marker}"):
            return p.parent
    return None


if IS_KAGGLE:
    # Kaggle mounts datasets read-only, but the vendored scripts write
    # predictions/weights relative to their own file location
    # (scripts/dataspec.py) -- copy the code to a writable location first.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "zarr>=3.0.10", "scipy", "tqdm", "polars", "pyscipopt",
            "tracksdata @ git+https://github.com/royerlab/tracksdata@main",
        ],
        check=True,
    )

    SRC_MOUNT = _find_mount([Path("/kaggle/input/tracking-cellmot-src")], "pyproject.toml")
    if SRC_MOUNT is None:
        raise FileNotFoundError(
            "tracking-cellmot-src dataset not found under /kaggle/input -- add it as a "
            "data source (see docs/0_coding_standards.md's 'Pushing Notebooks To Kaggle')."
        )
    REPO_ROOT = Path("/kaggle/working/repo")
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(SRC_MOUNT, REPO_ROOT)

    ARTIFACTS_MOUNT = _find_mount(
        [
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ],
        "weights",
    )
else:
    REPO_ROOT = Path.cwd().parent
    ARTIFACTS_MOUNT = None  # pretrained weights are Kaggle-only; train locally instead

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from dataspec import DATASET_PATH  # noqa: E402 -- needs REPO_ROOT on sys.path first

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = Path(f"/kaggle/input/competitions/{COMPETITION}")
TEST_DIR = COMP_DIR / "test" if IS_KAGGLE else REPO_ROOT / "data" / "test"

SEED = 0
RUN_MODE = "submission"  # "train" | "submission"

# --- "submission" mode: which weights to predict with ---------------------
USE_PRETRAINED = True  # True -> the public baseline checkpoint; False -> our own weights/ below
PRETRAINED_METHOD = "unet_transformer"
PRETRAINED_SPLIT = "0"

# --- "train" mode, and our-own-weights naming for "submission" mode -------
METHOD = "baseline"
SPLIT = "0"
EPOCHS = 3

# --- test-time detection/linking knobs (only used in "submission" mode) ---
# GT is sparse so the detector is poorly calibrated; ~0.99 scored best in the
# baseline author's sweep. ILP (global, flow-consistent linking) scored
# ~0.73 -> ~0.79 over the faster greedy linker in their notes.
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0


def run(*args: str) -> None:
    """Run a vendored script with the current kernel's interpreter."""
    subprocess.run([sys.executable, *args], check=True, cwd=REPO_ROOT)


def resolve_weights() -> tuple[Path, str]:
    """Return (checkpoint path, method name) per USE_PRETRAINED."""
    if USE_PRETRAINED:
        if ARTIFACTS_MOUNT is None:
            raise FileNotFoundError(
                "USE_PRETRAINED=True but the cellmot-baseline-artifacts dataset isn't "
                "mounted -- add it as a data source, or set USE_PRETRAINED=False to use "
                "our own weights/ (requires RUN_MODE='train' first)."
            )
        split_dir = ARTIFACTS_MOUNT / "weights" / PRETRAINED_METHOD / f"split_{PRETRAINED_SPLIT}"
        return split_dir / "edge_predictor_best.pth", PRETRAINED_METHOD
    split_dir = REPO_ROOT / "weights" / METHOD / f"split_{SPLIT}"
    return split_dir / "edge_predictor_best.pth", METHOD


print(f"IS_KAGGLE={IS_KAGGLE}  REPO_ROOT={REPO_ROOT}")
if IS_KAGGLE:
    print(f"ARTIFACTS_MOUNT={ARTIFACTS_MOUNT}")

## 2. Train (optional — skip if `USE_PRETRAINED`)

Only runs in `RUN_MODE == "train"`. Not needed for a first submission (see
`USE_PRETRAINED` above) — this is how to train our own checkpoint to try to
beat the public baseline later.

In [ ]:
if RUN_MODE == "train":
    run(
        "scripts/train_unet_transformer.py",
        "--split", SPLIT,
        "--epochs", str(EPOCHS),
    )
    print(f"Trained weights: {REPO_ROOT}/weights/{METHOD}/split_{SPLIT}/edge_predictor_best.pth")

*Insight: fill in after running — training loss curve, whether it converged
in `EPOCHS` epochs, any stability issues.*

## 3. Predict on the competition test set

Only runs in `RUN_MODE == "submission"`. `predict_unet_transformer.py`
requires a `dataset_splits.json` listing which videos to predict — the real
`test/` directory doesn't ship one (that's a train-only, fold-splitting
concept), so build a synthetic one-fold file listing every test video
first, matching the approach in the baseline author's own public inference
notebook (`thibautgoldsborough/unet-baseline-inference-submission`).

In [ ]:
if RUN_MODE == "submission":
    import json

    test_stems = sorted(p.stem for p in TEST_DIR.glob("*.zarr"))
    print(f"{len(test_stems)} test videos under {TEST_DIR}")

    test_splits_file = REPO_ROOT / "kaggle_test_splits.json"
    test_splits_file.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}]))

    weights_path, predict_method = resolve_weights()

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(TEST_DIR),
        "--splits", str(test_splits_file),
        "--split", "0",
        "--method", predict_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")

    run(*predict_args)

## 4. Build `submission.csv`

Flattens the predicted `.geff` graphs (one per test video) into the
competition's CSV schema — verified against the real `sample_submission.csv`
downloaded via the Kaggle CLI: `id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`,
one `node` row per detection and one `edge` row per link.

In [ ]:
if RUN_MODE == "submission":
    import os

    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    predictions_dir = REPO_ROOT / "predictions" / kaggle_user / predict_method / "split_0"
    submission_csv = Path("/kaggle/working/submission.csv") if IS_KAGGLE else REPO_ROOT / "submission.csv"

    run(
        "scripts/geffs_to_csv.py",
        "--in-dir", str(predictions_dir),
        "--csv", str(submission_csv),
    )
    print(f"Wrote {submission_csv}")

## 5. (Optional) Validate methodology on a train fold

The real `test/` set has no local ground truth to score against. To
sanity-check the weights/detection/linking config *before* spending a
submission attempt, predict on a held-out **train** fold instead (real GT
available) and score locally with the competition's own metric
(`docs/1_instructions.md` / `metrics.md`). Builds its own deterministic
90/10 train/val split the same way `train_unet_transformer.py` does when no
`dataset_splits.json` is present, so it doesn't depend on a prior run.

In [ ]:
VALIDATE_ON_TRAIN_FOLD = False  # set True to sanity-check before submitting

if VALIDATE_ON_TRAIN_FOLD:
    import json
    import random

    stems = sorted(
        p.name[:-5] for p in DATASET_PATH.glob("*.zarr")
        if (DATASET_PATH / f"{p.name[:-5]}.geff").exists()
    )
    random.Random(0).shuffle(stems)
    n_val = max(1, len(stems) // 10)
    train_splits_file = REPO_ROOT / "kaggle_train_splits.json"
    train_splits_file.write_text(json.dumps(
        [{"split": 0, "train": stems[n_val:], "test": stems[:n_val]}]
    ))
    print(f"{len(stems) - n_val} train / {n_val} val videos under {DATASET_PATH}")

    weights_path, validate_method = resolve_weights()

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(DATASET_PATH),
        "--splits", str(train_splits_file),
        "--split", "0",
        "--method", validate_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--evaluate",
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")

    run(*predict_args)

*Insight: fill in after running — edge Jaccard, division Jaccard, final
score printed by `--evaluate`, and how it compares to the previous best
(record in `README.md`'s "Current best result" table).*

## Submitting to Kaggle

This competition scores a **file upload**, not a notebook rerun (see
`docs/1_instructions.md`) — after `RUN_MODE == "submission"` finishes on
Kaggle, download `submission.csv` from the kernel's Output tab, then
either upload it on the competition's Submit page, or from a shell with
Kaggle CLI access:

```bash
uv run kaggle competitions submit \
    -c biohub-cell-tracking-during-development \
    -f submission.csv \
    -m "unet_transformer split_0 pretrained, ILP, det-threshold 0.99"
```

Not run automatically from this notebook — submissions count against a
daily quota and should be a deliberate action, not a side effect of
re-running a cell.

## Findings / limitations / next experiment

- **Findings**: _fill in after running._
- **Limitations**: the pretrained checkpoint (`unet_transformer`, split 0)
  wasn't trained to convergence per the baseline author's own notes — it's
  a starting point to beat, not a ceiling. `DET_THRESHOLD`/ILP weights
  above are their reported best settings, not necessarily ours.
- **Next**: once a first submission is banked, sweep `DET_THRESHOLD` /
  `ILP_*` weights via `VALIDATE_ON_TRAIN_FOLD` (cheap, no submission quota
  spent); then try training our own checkpoint (`RUN_MODE = "train"`,
  `USE_PRETRAINED = False`) longer / with architecture changes to actually
  beat the public baseline.